# 🌸 Iris Çiçeği — Keşifsel Veri Analizi (EDA)

Bu notebook, makine öğrenmesinin klasik veri seti **Iris** üzerinde temel bir Keşifsel Veri Analizi (EDA) gerçekleştirir.

**Adımlar:**
1. Veri Yükleme
2. Veri Keşfi (shape, describe, info, isnull)
3. Görselleştirme (Pairplot, Boxplot, Heatmap)
4. Model Eğitimi (K-Nearest Neighbors)


## ⚙️ Adım 0: Kütüphaneler

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Görsel stil ayarları
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100


## 📂 Adım 1: Veri Yükleme

Veri seti UCI Machine Learning Repository'den alınmıştır: [bezdekIris.data](https://archive.ics.uci.edu/ml/machine-learning-databases/iris/bezdekIris.data)


In [ ]:
# Eğer veri seti zip içindeyse aç
import os
if os.path.exists("iris.zip") and not os.path.exists("bezdekIris.data"):
    import zipfile
    with zipfile.ZipFile("iris.zip", "r") as z:
        z.extractall()

df = pd.read_csv(
    "bezdekIris.data",
    sep=",",
    header=None,
    names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
)

df.head()


### Sütun Açıklamaları

| Sütun | Açıklama |
|-------|----------|
| `sepal_length` | Çanak yaprak uzunluğu (cm) |
| `sepal_width` | Çanak yaprak genişliği (cm) |
| `petal_length` | Taç yaprak uzunluğu (cm) |
| `petal_width` | Taç yaprak genişliği (cm) |
| `species` | Hedef değişken: *Iris-setosa*, *Iris-versicolor*, *Iris-virginica* |


## 🔎 Adım 2: Veri Keşfi

In [ ]:
print("Boyut (satır x sütun):", df.shape)
print("\nSütun isimleri:", df.columns.tolist())
print("\nTür dağılımı:")
print(df["species"].value_counts())


In [ ]:
# Temel istatistikler
df.describe()


In [ ]:
# Veri tipleri ve bellek kullanımı
df.info()


In [ ]:
# Eksik değer kontrolü
missing = df.isnull().sum()
print("Eksik değer sayısı:")
print(missing)
print(f"\nToplamda {missing.sum()} eksik değer {'bulunmaktadır' if missing.sum() > 0 else 'bulunmamaktadır.'}")


## 📊 Adım 3: Görselleştirme

In [ ]:
# Tüm özelliklerin birbiriyle ilişkisini gösteren pairplot
sns.pairplot(df, hue="species", palette="Set2")
plt.suptitle("Özellikler Arası İlişkiler (Pairplot)", y=1.02, fontsize=13)
plt.show()


In [ ]:
# Türlere göre taç yaprak uzunluğu dağılımı
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="species",
    y="petal_length",
    hue="species",
    palette="Set2"
)
plt.title("Türlere Göre Taç Yaprak Uzunluğu Dağılımı (Boxplot)", fontsize=13, pad=15)
plt.xlabel("Çiçek Türü", fontsize=11)
plt.ylabel("Taç Yaprak Uzunluğu (cm)", fontsize=11)
plt.show()


In [ ]:
# Özellikler arası korelasyon heatmap
numeric_df = df.drop(columns=["species"])
korelasyon = numeric_df.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    data=korelasyon,
    annot=True,
    cmap="coolwarm",
    vmin=-1, vmax=1,
    fmt=".2f",
    linewidths=1
)
plt.title("Özellikler Arasındaki İlişki Derecesi (Heatmap)", fontsize=13, pad=15)
plt.show()


## 🤖 Adım 4: Model Eğitimi (K-Nearest Neighbors)

EDA'da gördüğümüz örüntüleri doğrulamak için basit bir KNN sınıflandırıcısı eğitiyoruz.


In [ ]:
# Özellikler ve hedef değişkeni ayır
X = df.drop(columns=["species"])
y = df["species"]

# Eğitim / test bölümü (%80 / %20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti:   {X_test.shape[0]} örnek")


In [ ]:
# KNN modelini eğit
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Tahmin ve doğruluk
y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Doğruluğu: {acc:.2%}\n")
print(classification_report(y_test, y_pred))


In [ ]:
# Karmaşıklık matrisi
cm = confusion_matrix(y_test, y_pred, labels=knn.classes_)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=knn.classes_,
    yticklabels=knn.classes_
)
plt.title("Karmaşıklık Matrisi (Confusion Matrix)", fontsize=13)
plt.ylabel("Gerçek")
plt.xlabel("Tahmin")
plt.tight_layout()
plt.show()
